# AU-FISH Multi-Seed Individual Run

**Protocol:** `time_based`  
**Model:** `Faster R-CNN ResNet50-FPN`  
**Seed:** `123`  
**Experiment tag:** `multiseed_fullset_dedup_v1`

Notebook ini hanya menangani satu training run. Semua output disimpan pada folder
protocol, model, dan seed yang spesifik sehingga tidak bercampur dengan run lain.

Split dataset tidak diubah antar-seed. Preprocessing final tetap menggunakan:

- explicit image lists;
- filtered two-class annotations: coral and fish;
- exact duplicate removal yang telah dilakukan sebelumnya;
- minimum box width/height 5 px;
- minimum box area 25 px²;
- common filtered COCO ground truth.

Jangan mengubah protocol, model, seed, split, atau output root di tengah eksekusi.

In [1]:
# CELL 1 — INSTALL DEPENDENCIES
!pip -q install pycocotools pandas pillow tqdm

In [2]:
# CELL 2 — MOUNT GOOGLE DRIVE
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# CELL 3 — FIXED RUN CONFIGURATION
import contextlib
import hashlib
import io
import json
import os
import platform
import random
import shutil
import sys
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
)
from torchvision.models.detection.faster_rcnn import (
    FastRCNNPredictor,
)
from torchvision.transforms import functional as F_tv
from tqdm.auto import tqdm

PROTOCOL = "time_based"
MODEL_KEY = "faster_rcnn"
MODEL_LABEL = "Faster R-CNN ResNet50-FPN"
SEED = 123

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/au-fish/dataset_v2"
)
DATASET_ROOT = PROJECT_ROOT / PROTOCOL

TRAIN_DIR = DATASET_ROOT / "train"
VAL_DIR = DATASET_ROOT / "val"
TEST_DIR = DATASET_ROOT / "test"

INDIVIDUAL_TAG = "multiseed_fullset_dedup_v1"
LEGACY_FINAL_TAG = "simple_wbf_fullset_dedup_v1"

EXPERIMENT_ROOT = (
    DATASET_ROOT
    / "experiment_logs"
    / INDIVIDUAL_TAG
)

RUN_ROOT = (
    EXPERIMENT_ROOT
    / "individual"
    / MODEL_KEY
    / f"seed_{SEED}"
)

TRAIN_RUN_DIR = (
    DATASET_ROOT
    / "runs"
    / INDIVIDUAL_TAG
    / MODEL_KEY
    / f"seed_{SEED}"
)

RAW_DIR = RUN_ROOT / "raw_predictions"
RESULTS_DIR = RUN_ROOT / "results"
MANIFEST_DIR = RUN_ROOT / "manifests"
CHECKPOINT_DIR = TRAIN_RUN_DIR / "checkpoints"
SHARED_GT_DIR = EXPERIMENT_ROOT / "shared_ground_truth"

for path in [
    RAW_DIR,
    RESULTS_DIR,
    MANIFEST_DIR,
    CHECKPOINT_DIR,
    SHARED_GT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

BEST_CHECKPOINT = (
    CHECKPOINT_DIR
    / "best_by_validation_map.pth"
)
LAST_CHECKPOINT = (
    CHECKPOINT_DIR
    / "last.pth"
)

CLASS_NAMES = ["coral", "fish"]
CLASS_NAMES_FRCNN = [
    "__background__",
    "coral",
    "fish",
]
KEEP_CLASSES = set(CLASS_NAMES)
NUM_CLASSES = 3

MIN_BOX_SIZE = 5.0
MIN_BOX_AREA = 25.0

NUM_EPOCHS = 10
TRAIN_BATCH = 2
EVAL_BATCH = 2
NUM_WORKERS = 2

LEARNING_RATE = 0.0005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
STEP_SIZE = 8
GAMMA = 0.1
GRADIENT_CLIP = 5.0

SCORE_THRESHOLD = 0.001
MAX_DETECTIONS_PER_IMAGE = 300

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

FORCE_RETRAIN = False
FORCE_REINFERENCE = False

EXPECTED_IMAGES = {
    "time_based": {"train": 1061, "val": 203, "test": 282},
    "source_based": {"train": 1083, "val": 232, "test": 231},
}

print("=" * 80)
print("AU-FISH MULTI-SEED INDIVIDUAL RUN")
print("Protocol :", PROTOCOL)
print("Model    :", MODEL_LABEL)
print("Seed     :", SEED)
print("Run root :", RUN_ROOT)
print("Device   :", DEVICE)
print("=" * 80)

AU-FISH MULTI-SEED INDIVIDUAL RUN
Protocol : time_based
Model    : Faster R-CNN ResNet50-FPN
Seed     : 123
Run root : /content/drive/MyDrive/au-fish/dataset_v2/time_based/experiment_logs/multiseed_fullset_dedup_v1/individual/faster_rcnn/seed_123
Device   : cuda


In [4]:
# CELL 4 — REPRODUCIBILITY AND FILE HELPERS
def set_global_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = (
        torch.initial_seed() % 2**32
    )

    np.random.seed(worker_seed)
    random.seed(worker_seed)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temporary = path.with_suffix(
        path.suffix + ".tmp"
    )

    with open(
        temporary,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            data,
            file,
            ensure_ascii=False,
            indent=2,
        )

    temporary.replace(path)


set_global_seed(SEED)

generator = torch.Generator()
generator.manual_seed(SEED)

run_config = {
    "protocol": PROTOCOL,
    "model": MODEL_KEY,
    "seed": SEED,
    "experiment_tag": INDIVIDUAL_TAG,
    "epochs": NUM_EPOCHS,
    "train_batch": TRAIN_BATCH,
    "eval_batch": EVAL_BATCH,
    "optimizer": "SGD",
    "learning_rate": LEARNING_RATE,
    "momentum": MOMENTUM,
    "weight_decay": WEIGHT_DECAY,
    "scheduler": "StepLR",
    "scheduler_step_size": STEP_SIZE,
    "scheduler_gamma": GAMMA,
    "gradient_clip_max_norm": GRADIENT_CLIP,
    "explicit_photometric_augmentation": False,
    "explicit_geometric_augmentation": False,
    "checkpoint_selection": (
        "highest validation mAP50-95; "
        "validation mAP50 tie-break; "
        "lower training loss final tie-break"
    ),
    "raw_prediction_confidence": SCORE_THRESHOLD,
    "max_detections_per_image": (
        MAX_DETECTIONS_PER_IMAGE
    ),
}

save_json(
    run_config,
    MANIFEST_DIR / "run_config.json",
)

print("Global seed set:", SEED)

Global seed set: 123


In [5]:
# CELL 5 — DATASET AND COMMON GROUND-TRUTH PREFLIGHT
for split_name, split_dir in {
    "train": TRAIN_DIR,
    "val": VAL_DIR,
    "test": TEST_DIR,
}.items():
    annotation_path = (
        split_dir / "annotations.json"
    )

    if not annotation_path.exists():
        raise FileNotFoundError(
            f"Missing annotations: {annotation_path}"
        )

    coco = load_json(annotation_path)
    expected = EXPECTED_IMAGES[PROTOCOL][
        split_name
    ]

    if len(coco["images"]) != expected:
        raise RuntimeError(
            f"{split_name}: expected {expected} images, "
            f"found {len(coco['images'])}"
        )

    missing_images = []

    for image_info in coco["images"]:
        image_path = (
            split_dir
            / "images"
            / image_info["file_name"]
        )

        if not image_path.exists():
            missing_images.append(
                str(image_path)
            )

    if missing_images:
        raise FileNotFoundError(
            f"{split_name}: "
            f"{len(missing_images)} missing images. "
            f"First: {missing_images[0]}"
        )

    print(
        f"✅ {split_name}: "
        f"{len(coco['images'])} images"
    )

legacy_gt_dir = (
    DATASET_ROOT
    / "experiment_logs"
    / LEGACY_FINAL_TAG
    / "shared_ground_truth"
)

for file_name in [
    "val_gt_filtered.json",
    "test_gt_filtered.json",
]:
    source_path = legacy_gt_dir / file_name
    target_path = SHARED_GT_DIR / file_name

    if not source_path.exists():
        raise FileNotFoundError(
            f"Locked common GT not found: {source_path}"
        )

    if not target_path.exists():
        shutil.copy2(source_path, target_path)

    print("✅ Shared GT:", target_path)

✅ train: 1061 images
✅ val: 203 images
✅ test: 282 images
✅ Shared GT: /content/drive/MyDrive/au-fish/dataset_v2/time_based/experiment_logs/multiseed_fullset_dedup_v1/shared_ground_truth/val_gt_filtered.json
✅ Shared GT: /content/drive/MyDrive/au-fish/dataset_v2/time_based/experiment_logs/multiseed_fullset_dedup_v1/shared_ground_truth/test_gt_filtered.json


In [6]:
# CELL 6 — CLEAN TWO-CLASS COCO DATASET
class AUFishCocoDataset(Dataset):
    def __init__(
        self,
        split_dir,
        min_box_size=MIN_BOX_SIZE,
        min_box_area=MIN_BOX_AREA,
    ):
        self.split_dir = Path(split_dir)
        self.images_dir = (
            self.split_dir / "images"
        )

        coco = load_json(
            self.split_dir
            / "annotations.json"
        )

        self.images = coco["images"]
        self.categories = coco["categories"]

        self.category_name = {
            int(category["id"]): category["name"]
            for category in self.categories
        }

        self.class_to_label = {
            "coral": 1,
            "fish": 2,
        }

        self.annotations_by_image = defaultdict(
            list
        )

        for annotation in coco["annotations"]:
            self.annotations_by_image[
                int(annotation["image_id"])
            ].append(annotation)

        self.min_box_size = float(
            min_box_size
        )
        self.min_box_area = float(
            min_box_area
        )


    def __len__(self):
        return len(self.images)


    def __getitem__(self, index):
        image_info = self.images[index]
        image_id = int(image_info["id"])

        image_path = (
            self.images_dir
            / image_info["file_name"]
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        image_width = int(
            image_info["width"]
        )
        image_height = int(
            image_info["height"]
        )

        boxes = []
        labels = []
        areas = []
        iscrowd = []

        for annotation in (
            self.annotations_by_image[
                image_id
            ]
        ):
            class_name = self.category_name[
                int(annotation["category_id"])
            ]

            if class_name not in KEEP_CLASSES:
                continue

            x, y, width, height = map(
                float,
                annotation["bbox"],
            )

            if width <= 0 or height <= 0:
                continue

            x1 = max(0.0, x)
            y1 = max(0.0, y)
            x2 = min(
                float(image_width),
                x + width,
            )
            y2 = min(
                float(image_height),
                y + height,
            )

            box_width = x2 - x1
            box_height = y2 - y1
            box_area = box_width * box_height

            if (
                box_width < self.min_box_size
                or box_height < self.min_box_size
                or box_area < self.min_box_area
            ):
                continue

            boxes.append(
                [x1, y1, x2, y2]
            )
            labels.append(
                self.class_to_label[
                    class_name
                ]
            )
            areas.append(box_area)
            iscrowd.append(
                int(
                    annotation.get(
                        "iscrowd",
                        0,
                    )
                )
            )

        if boxes:
            boxes_tensor = torch.tensor(
                boxes,
                dtype=torch.float32,
            )
            labels_tensor = torch.tensor(
                labels,
                dtype=torch.int64,
            )
            areas_tensor = torch.tensor(
                areas,
                dtype=torch.float32,
            )
            iscrowd_tensor = torch.tensor(
                iscrowd,
                dtype=torch.int64,
            )
        else:
            boxes_tensor = torch.zeros(
                (0, 4),
                dtype=torch.float32,
            )
            labels_tensor = torch.zeros(
                (0,),
                dtype=torch.int64,
            )
            areas_tensor = torch.zeros(
                (0,),
                dtype=torch.float32,
            )
            iscrowd_tensor = torch.zeros(
                (0,),
                dtype=torch.int64,
            )

        target = {
            "boxes": boxes_tensor,
            "labels": labels_tensor,
            "image_id": torch.tensor(
                image_id,
                dtype=torch.int64,
            ),
            "area": areas_tensor,
            "iscrowd": iscrowd_tensor,
        }

        return F_tv.to_tensor(image), target


def collate_fn(batch):
    return tuple(zip(*batch))


train_dataset = AUFishCocoDataset(
    TRAIN_DIR
)
val_dataset = AUFishCocoDataset(
    VAL_DIR
)
test_dataset = AUFishCocoDataset(
    TEST_DIR
)

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=torch.cuda.is_available(),
)

print("Train:", len(train_dataset))
print("Val  :", len(val_dataset))
print("Test :", len(test_dataset))

Train: 1061
Val  : 203
Test : 282


In [7]:
# CELL 7 — MODEL AND COCOEVAL HELPERS
def get_model():
    model = fasterrcnn_resnet50_fpn(
        weights="DEFAULT"
    )

    in_features = (
        model.roi_heads
        .box_predictor
        .cls_score
        .in_features
    )

    model.roi_heads.box_predictor = (
        FastRCNNPredictor(
            in_features,
            NUM_CLASSES,
        )
    )

    model.roi_heads.detections_per_img = (
        MAX_DETECTIONS_PER_IMAGE
    )

    return model


def gt_path_for(split_label):
    file_name = (
        "val_gt_filtered.json"
        if split_label == "validation"
        else "test_gt_filtered.json"
    )

    return SHARED_GT_DIR / file_name


def predictions_to_coco_rows(raw):
    rows = []

    for record in raw.values():
        for box, score, label in zip(
            record["boxes"],
            record["scores"],
            record["labels"],
        ):
            x1, y1, x2, y2 = map(
                float,
                box,
            )

            width = x2 - x1
            height = y2 - y1

            if width <= 0 or height <= 0:
                continue

            rows.append({
                "image_id": int(
                    record["image_id"]
                ),
                "category_id": int(label) + 1,
                "bbox": [
                    x1,
                    y1,
                    width,
                    height,
                ],
                "score": float(score),
            })

    return rows


def evaluate_common_coco(gt_path, raw):
    rows = predictions_to_coco_rows(raw)

    if not rows:
        return {
            "mAP50_95": 0.0,
            "mAP50": 0.0,
            "mAP75": 0.0,
            "AP_coral": 0.0,
            "AP_fish": 0.0,
            "worst_class_AP": 0.0,
            "detections": 0,
        }

    with contextlib.redirect_stdout(
        io.StringIO()
    ):
        coco_gt = COCO(str(gt_path))
        coco_dt = coco_gt.loadRes(rows)

        evaluator = COCOeval(
            coco_gt,
            coco_dt,
            "bbox",
        )
        evaluator.evaluate()
        evaluator.accumulate()
        evaluator.summarize()

    class_ap = {}

    for category_id, class_name in [
        (1, "coral"),
        (2, "fish"),
    ]:
        with contextlib.redirect_stdout(
            io.StringIO()
        ):
            class_eval = COCOeval(
                coco_gt,
                coco_dt,
                "bbox",
            )
            class_eval.params.catIds = [
                category_id
            ]
            class_eval.evaluate()
            class_eval.accumulate()
            class_eval.summarize()

        class_ap[class_name] = float(
            class_eval.stats[0]
        )

    return {
        "mAP50_95": float(
            evaluator.stats[0]
        ),
        "mAP50": float(
            evaluator.stats[1]
        ),
        "mAP75": float(
            evaluator.stats[2]
        ),
        "AP_coral": class_ap["coral"],
        "AP_fish": class_ap["fish"],
        "worst_class_AP": min(
            class_ap["coral"],
            class_ap["fish"],
        ),
        "detections": len(rows),
    }


def predict_raw(
    model,
    data_loader,
    dataset,
    score_threshold=SCORE_THRESHOLD,
):
    model.eval()
    raw = {}

    with torch.no_grad():
        for images, targets in tqdm(
            data_loader,
            desc="Inference",
        ):
            device_images = [
                image.to(DEVICE)
                for image in images
            ]

            outputs = model(device_images)

            for target, output in zip(
                targets,
                outputs,
            ):
                image_id = int(
                    target["image_id"].item()
                )

                image_info = next(
                    image
                    for image in dataset.images
                    if int(image["id"]) == image_id
                )

                record = {
                    "image_id": image_id,
                    "file_name": str(
                        image_info["file_name"]
                    ),
                    "width": int(
                        image_info["width"]
                    ),
                    "height": int(
                        image_info["height"]
                    ),
                    "boxes": [],
                    "scores": [],
                    "labels": [],
                }

                boxes = (
                    output["boxes"]
                    .detach()
                    .cpu()
                    .numpy()
                )
                scores = (
                    output["scores"]
                    .detach()
                    .cpu()
                    .numpy()
                )
                labels = (
                    output["labels"]
                    .detach()
                    .cpu()
                    .numpy()
                )

                kept = 0

                for box, score, label in zip(
                    boxes,
                    scores,
                    labels,
                ):
                    if score < score_threshold:
                        continue

                    if kept >= (
                        MAX_DETECTIONS_PER_IMAGE
                    ):
                        break

                    x1, y1, x2, y2 = map(
                        float,
                        box,
                    )

                    if x2 <= x1 or y2 <= y1:
                        continue

                    # Convert FRCNN labels 1/2 to
                    # common raw labels 0/1.
                    common_label = int(label) - 1

                    if common_label not in {0, 1}:
                        continue

                    record["boxes"].append(
                        [x1, y1, x2, y2]
                    )
                    record["scores"].append(
                        float(score)
                    )
                    record["labels"].append(
                        common_label
                    )

                    kept += 1

                raw[str(image_id)] = record

    if len(raw) != len(dataset):
        raise RuntimeError(
            f"Prediction coverage mismatch: "
            f"{len(raw)} != {len(dataset)}"
        )

    return raw

In [8]:
# CELL 8 — TRAIN WITH VALIDATION-BASED CHECKPOINT SELECTION
def train_one_epoch(
    model,
    optimizer,
    data_loader,
    epoch,
):
    model.train()

    total_loss = 0.0
    completed_batches = 0
    skipped_batches = 0

    for images, targets in tqdm(
        data_loader,
        desc=f"Train epoch {epoch}",
    ):
        images = [
            image.to(DEVICE)
            for image in images
        ]

        targets = [
            {
                key: value.to(DEVICE)
                for key, value in target.items()
            }
            for target in targets
        ]

        loss_dict = model(
            images,
            targets,
        )

        losses = sum(
            loss
            for loss in loss_dict.values()
        )

        if not torch.isfinite(losses):
            skipped_batches += 1
            optimizer.zero_grad(
                set_to_none=True
            )
            continue

        optimizer.zero_grad(
            set_to_none=True
        )
        losses.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRADIENT_CLIP,
        )

        optimizer.step()

        total_loss += float(
            losses.item()
        )
        completed_batches += 1

    mean_loss = (
        total_loss
        / max(1, completed_batches)
    )

    return (
        mean_loss,
        skipped_batches,
    )


if BEST_CHECKPOINT.exists() and not FORCE_RETRAIN:
    print("Checkpoint already exists. Training skipped:")
    print(BEST_CHECKPOINT)
else:
    model = get_model().to(DEVICE)

    trainable_parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]

    optimizer = torch.optim.SGD(
        trainable_parameters,
        lr=LEARNING_RATE,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=STEP_SIZE,
        gamma=GAMMA,
    )

    best_key = None
    history = []

    for epoch in range(
        1,
        NUM_EPOCHS + 1,
    ):
        train_loss, skipped = train_one_epoch(
            model,
            optimizer,
            train_loader,
            epoch,
        )

        scheduler.step()

        validation_raw = predict_raw(
            model,
            val_loader,
            val_dataset,
        )

        validation_metrics = (
            evaluate_common_coco(
                gt_path_for("validation"),
                validation_raw,
            )
        )

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "skipped_batches": skipped,
            "learning_rate": optimizer.param_groups[0]["lr"],
            **{
                f"validation_{key}": value
                for key, value
                in validation_metrics.items()
            },
        }

        history.append(row)

        selection_key = (
            validation_metrics[
                "mAP50_95"
            ],
            validation_metrics[
                "mAP50"
            ],
            -train_loss,
        )

        if (
            best_key is None
            or selection_key > best_key
        ):
            best_key = selection_key

            torch.save(
                {
                    "model_state_dict": (
                        model.state_dict()
                    ),
                    "epoch": epoch,
                    "seed": SEED,
                    "protocol": PROTOCOL,
                    "selection_key": (
                        selection_key
                    ),
                    "validation_metrics": (
                        validation_metrics
                    ),
                    "run_config": run_config,
                },
                BEST_CHECKPOINT,
            )

            print(
                "✅ New best checkpoint:",
                BEST_CHECKPOINT,
            )

        torch.save(
            {
                "model_state_dict": (
                    model.state_dict()
                ),
                "epoch": epoch,
                "seed": SEED,
                "protocol": PROTOCOL,
                "run_config": run_config,
            },
            LAST_CHECKPOINT,
        )

        pd.DataFrame(history).to_csv(
            RESULTS_DIR
            / "training_history.csv",
            index=False,
        )

        print(
            f"Epoch {epoch}: "
            f"loss={train_loss:.6f}, "
            f"val mAP50-95="
            f"{validation_metrics['mAP50_95']:.6f}"
        )

if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Best checkpoint not produced: "
        f"{BEST_CHECKPOINT}"
    )

print("✅ Best checkpoint:", BEST_CHECKPOINT)
print("SHA256:", sha256_file(BEST_CHECKPOINT))

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 254MB/s]


Train epoch 1:   0%|          | 0/531 [00:00<?, ?it/s]

Inference:   0%|          | 0/102 [00:00<?, ?it/s]

Exception ignored in: ^<function _MultiProcessingDataLoaderIter.__del__ at 0x7d51ea700e00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d51ea700e00>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

...

Inference:   0%|          | 0/102 [00:00<?, ?it/s]

✅ New best checkpoint: /content/drive/MyDrive/au-fish/dataset_v2/time_based/runs/multiseed_fullset_dedup_v1/faster_rcnn/seed_123/checkpoints/best_by_validation_map.pth
Epoch 9: loss=0.164613, val mAP50-95=0.422849


Train epoch 10:   0%|          | 0/531 [00:00<?, ?it/s]

Inference:   0%|          | 0/102 [00:00<?, ?it/s]

✅ New best checkpoint: /content/drive/MyDrive/au-fish/dataset_v2/time_based/runs/multiseed_fullset_dedup_v1/faster_rcnn/seed_123/checkpoints/best_by_validation_map.pth
Epoch 10: loss=0.160627, val mAP50-95=0.423086
✅ Best checkpoint: /content/drive/MyDrive/au-fish/dataset_v2/time_based/runs/multiseed_fullset_dedup_v1/faster_rcnn/seed_123/checkpoints/best_by_validation_map.pth
SHA256: 758b7cbd1b2279eb4f151b2d698a2a94cd6f1aadaae9a784e2c79b9bf40c8501


In [9]:
# CELL 9 — LOCKED CHECKPOINT VALIDATION AND TEST INFERENCE
checkpoint = torch.load(
    BEST_CHECKPOINT,
    map_location=DEVICE,
)

evaluation_model = get_model().to(
    DEVICE
)
evaluation_model.load_state_dict(
    checkpoint["model_state_dict"]
)
evaluation_model.eval()

result_rows = []

split_objects = {
    "validation": (
        val_loader,
        val_dataset,
    ),
    "test": (
        test_loader,
        test_dataset,
    ),
}

for split_label, (
    data_loader,
    dataset,
) in split_objects.items():
    output_path = (
        RAW_DIR
        / f"{split_label}_{MODEL_KEY}.json"
    )

    if (
        output_path.exists()
        and not FORCE_REINFERENCE
    ):
        raw = load_json(output_path)

        if len(raw) != len(dataset):
            raise RuntimeError(
                f"Existing raw prediction coverage mismatch: "
                f"{len(raw)} != {len(dataset)}"
            )

        print(
            "Raw prediction already exists:",
            output_path,
        )
    else:
        raw = predict_raw(
            evaluation_model,
            data_loader,
            dataset,
        )

        save_json(
            raw,
            output_path,
        )

    metrics = evaluate_common_coco(
        gt_path_for(split_label),
        raw,
    )

    result_rows.append({
        "protocol": PROTOCOL,
        "model": MODEL_KEY,
        "seed": SEED,
        "selected_epoch": checkpoint[
            "epoch"
        ],
        "split": split_label,
        **metrics,
    })

result_df = pd.DataFrame(
    result_rows
)

result_df.to_csv(
    RESULTS_DIR
    / "common_cocoeval_results.csv",
    index=False,
)

display(result_df)

Inference:   0%|          | 0/102 [00:00<?, ?it/s]

Inference:   0%|          | 0/141 [00:00<?, ?it/s]

,protocol,model,seed,selected_epoch,split,mAP50_95,mAP50,mAP75,AP_coral,AP_fish,worst_class_AP,detections
0,time_based,faster_rcnn,123,10,validation,0.423086,0.698841,0.438105,0.500177,0.345996,0.345996,2387
1,time_based,faster_rcnn,123,10,test,0.371855,0.633213,0.365787,0.430272,0.313438,0.313438,3856


In [10]:
# CELL 10 — CHECKPOINT AND OUTPUT MANIFEST
manifest = {
    "protocol": PROTOCOL,
    "model": MODEL_KEY,
    "seed": SEED,
    "selected_epoch": int(
        checkpoint["epoch"]
    ),
    "checkpoint_path": str(
        BEST_CHECKPOINT
    ),
    "checkpoint_sha256": sha256_file(
        BEST_CHECKPOINT
    ),
    "validation_raw_predictions": str(
        RAW_DIR
        / "validation_faster_rcnn.json"
    ),
    "test_raw_predictions": str(
        RAW_DIR
        / "test_faster_rcnn.json"
    ),
    "training_history": str(
        RESULTS_DIR
        / "training_history.csv"
    ),
    "common_cocoeval_results": str(
        RESULTS_DIR
        / "common_cocoeval_results.csv"
    ),
    "checkpoint_selection": (
        run_config[
            "checkpoint_selection"
        ]
    ),
}

save_json(
    manifest,
    MANIFEST_DIR
    / "checkpoint_manifest.json",
)

required_outputs = [
    Path(manifest["checkpoint_path"]),
    Path(manifest["validation_raw_predictions"]),
    Path(manifest["test_raw_predictions"]),
    Path(manifest["training_history"]),
    Path(manifest["common_cocoeval_results"]),
]

missing = [
    str(path)
    for path in required_outputs
    if not path.exists()
]

if missing:
    raise RuntimeError(
        "Missing final outputs:\n"
        + "\n".join(missing)
    )

print("✅ Individual multi-seed run complete.")
print("Run root:", RUN_ROOT)
print("Outputs are isolated by protocol/model/seed.")

✅ Individual multi-seed run complete.
Run root: /content/drive/MyDrive/au-fish/dataset_v2/time_based/experiment_logs/multiseed_fullset_dedup_v1/individual/faster_rcnn/seed_123
Outputs are isolated by protocol/model/seed.


## Important methodological note

Faster R-CNN multi-seed runs supersede the earlier retained legacy checkpoint
for multi-seed analysis. All seeds use the clean deduplicated annotations and
select the checkpoint by validation mAP50-95, with validation mAP50 and lower
training loss as tie-breakers.

The training budget remains 10 epochs and explicit photometric/geometric
augmentation remains disabled to preserve continuity with the earlier setup.